# Eksperimen 23 v2: Pure End-to-End HeteroGraphSAGE Classification (Tanpa XGBoost)

Notebook ini mengimplementasikan eksperimen `EXP23 v2`:
- Partisi rolling window 2016-2020: 300 hari Train, 30 hari Val, 8 hari Embargo Gap, 60 hari Test (SHINE FRAUD II Alignment).
- Pure End-to-End HeteroGraphSAGE Classifier (Ablasi Eliminasi XGBoost).
- **Hyperparameter v2**: Hidden Dim 256, 2 Layer GraphSAGE (Mean Aggr), Neighbor Sampling [25, 10], Batch Size 1024.
- **Optimasi v2**: LR 0.001 (Adam, Decay 0.5x / 5 epoch), Early Stopping Patience 15 Epochs tanpa peningkatan AUPRC.
- Klasifikasi langsung melalui linear classifier head bawaan model GNN ($p_i = \sigma(W h_i + b)$).
- Kalibrasi threshold dinamis $\tau^*$ langsung pada probabilitas kontinu GNN pada Validation Set.
- Komparasi performa murni 3 metode neighbor sampling: Uniform, Topology-Aware, dan Importance-Based.


In [ ]:
%%writefile config.kaggle.yaml
data:
  transactions: "/kaggle/input/credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016

time_window:
  reference_date: "2018-01-01" # Tanggal T0 (era modern 2016-2020 dengan sebaran fraud stabil)
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60

sampling:
  method: "uniform" # Pilihan: uniform, topology, importance
  undersampling_ratio: 1.0 # 1.0 = tidak ada undersampling (mempertahankan topologi asli)

model:
  hidden_channels: 256 # Sesuai gambar: Hidden Dimension 256 (ReLU & BatchNorm)
  out_channels: 1
  num_layers: 2 # Sesuai gambar: Lapisan GraphSAGE 2
  aggr: "mean" # Sesuai gambar: Mean Aggregator
  dropout: 0.2 # Regularisasi dropout optimal (v5 Golden Balance)

training:
  batch_size: 1024 # Sesuai gambar: Ukuran Batch 1024
  epochs: 100 # Maksimal 100 epoch dengan early stopping
  learning_rate: 0.001 # Sesuai gambar: Learning Rate 0.001 (Adam)
  weight_decay: 0.0001 # Regularisasi L2 seimbang pada Adam optimizer (v5)
  label_smoothing: 0.0 # Dinonaktifkan: Mencegah tabrakan matematis dengan pos_weight & mode collapse (v5)
  grad_clip_norm: 5.0 # Mencegah exploding gradient tanpa memotong sinyal fraud alami (v5)
  warmup_epochs: 0 # Pembelajaran aktif sejak epoch pertama (v5)
  use_lr_scheduler: true # Kurangi LR saat performa validasi mendatar
  lr_scheduler_factor: 0.5 # Sesuai gambar: Decay 0.5x
  lr_scheduler_patience: 5 # Sesuai gambar: Decay per 5 epoch
  early_stopping_patience: 15 # Sesuai gambar: Patience 15 epoch tanpa peningkatan AUPRC
  loss_weight_strategy: "sqrt" # Pilihan: sqrt (teredam ~27x), clipped (maks batas), full (rasio penuh ~732x)
  max_pos_weight: 50.0 # Batas clipping pos_weight jika memilih strategi clipped
  log_step_interval: 1 # Interval pencatatan per-step mini-batch (1 = catat setiap step untuk kurva halus)
  num_neighbors: [25, 10] # Sesuai gambar: Ukuran Sampel (k) 25 & 10 (Hop-1 dan Hop-2)

output:
  model_dir: "/kaggle/working/model/exp23/v2"
  result_dir: "/kaggle/working/result/exp23/v2"



In [ ]:
%%writefile config.py
import yaml
from pathlib import Path

def load_config(config_path: str = 'config.reference.yaml') -> dict:
    """
    Memuat file konfigurasi YAML untuk EXP23 (Pure End-to-End HeteroGraphSAGE).
    """
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Konfigurasi tidak ditemukan di: {path.resolve()}")
        
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
        
    return config



In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pathlib import Path

def load_and_preprocess_data(file_path: str, start_year: int = 2016, nrows: int = None) -> tuple:
    """
    Memuat dataset IBM Credit Card Transactions dan memfilter era modern (Year >= start_year).
    Melakukan parsing timestamp kronologis, pembersihan nominal Amount, encoding target,
    dan transformasi kategorikal.
    """
    print(f"Loading data from {file_path} (Filter: Year >= {start_year}, nrows={nrows})...")
    
    cat_columns = ['Use Chip', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?']
    
    # Jika start_year ditentukan dan nrows None/besar, gunakan chunking efisien untuk melewati tahun 1991-2015
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000
        
        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            # Filter baris dengan tahun >= start_year
            filtered_chunk = chunk[chunk['Year'] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break
                    
        if not chunks:
            raise ValueError(f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}")
            
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"Memproses {len(df)} baris data...")
    print("Parsing timestamps...")
    df['Time'] = df['Time'].fillna('00:00')
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + 
        df['Month'].astype(str).str.zfill(2) + '-' + 
        df['Day'].astype(str).str.zfill(2) + ' ' + 
        df['Time']
    )
    
    print("Sorting chronologically...")
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print("Cleaning Amount...")
    df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    
    print("Encoding target label...")
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})
    
    print("Encoding categorical features...")
    for col in cat_columns:
        df[col] = df[col].fillna('<missing>').astype(str)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        
    # Scale Amount menggunakan StandardScaler
    scaler = StandardScaler()
    df['Amount'] = scaler.fit_transform(df[['Amount']])

    # Fitur numerik/kategori untuk node Transaction
    feature_cols = ['Amount'] + cat_columns
    
    return df, feature_cols

def get_temporal_window_split(
    df: pd.DataFrame,
    reference_date: str = '2018-01-01',
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60
) -> tuple:
    """
    Membagi dataset berdasarkan protokol rolling window SHINE FRAUD II:
    - Train Window      : [T0, T0 + train_days)
    - Validation Window : [T0 + train_days, T0 + train_days + val_days)
    - Embargo Gap       : [T0 + train_days + val_days, T0 + train_days + val_days + gap_days)
    - Test Window       : [T0 + train_days + val_days + gap_days, T0 + train_days + val_days + gap_days + test_days)
    
    Mengembalikan: (train_df, val_df, test_df, gap_df, stats_dict)
    """
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)
    
    train_df = df[(df['datetime'] >= t0) & (df['datetime'] < t_val_start)].reset_index(drop=True)
    val_df = df[(df['datetime'] >= t_val_start) & (df['datetime'] < t_gap_start)].reset_index(drop=True)
    gap_df = df[(df['datetime'] >= t_gap_start) & (df['datetime'] < t_test_start)].reset_index(drop=True)
    test_df = df[(df['datetime'] >= t_test_start) & (df['datetime'] < t_test_end)].reset_index(drop=True)
    
    def compute_split_stats(name: str, split_df: pd.DataFrame, start_dt, end_dt):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0,
                "fraud_rate_pct": 0.0,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0
            }
        n_fraud = int(split_df['Is Fraud?'].sum())
        n_users = int(split_df['User'].nunique())
        n_merchants = int(split_df['Merchant Name'].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4),
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)
    
    stats_dict = {
        "protocol": "SHINE FRAUD II Rolling Window Alignment",
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats("Validation", val_df, t_val_start, t_gap_start),
            "embargo_gap": compute_split_stats("Embargo Gap", gap_df, t_gap_start, t_test_start),
            "test": compute_split_stats("Test", test_df, t_test_start, t_test_end),
            "total_window": compute_split_stats("Total Window", total_window_df, t0, t_test_end)
        }
    }
    
    return train_df, val_df, test_df, gap_df, stats_dict

def generate_rolling_window_indices(
    df: pd.DataFrame,
    start_date: str = '2016-01-01',
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60,
    step_days: int = 30
):
    """
    Generator untuk memproduksi beberapa timeframe jendela bergulir (rolling windows).
    """
    total_window_span = timedelta(days=train_days + val_days + gap_days + test_days)
    curr_t0 = pd.to_datetime(start_date)
    max_datetime = df['datetime'].max()
    
    fold_idx = 0
    while curr_t0 + total_window_span <= max_datetime:
        fold_idx += 1
        yield fold_idx, curr_t0.strftime('%Y-%m-%d')
        curr_t0 += timedelta(days=step_days)



In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler

def graph_level_undersample(df: pd.DataFrame, ratio: float = 1.0, seed: int = 42) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp21, exp23, & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df
        
    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(df: pd.DataFrame, feature_cols: list) -> HeteroData:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi tabular (feature_cols) & label target
      - user: Agregat rata-rata Amount & jumlah transaksi (scaled)
      - merchant: Agregat rata-rata Amount & jumlah transaksi (scaled)
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)
    """
    print(f"Membangun HeteroData graf ({len(df)} transaksi)...")
    data = HeteroData()
    
    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # 2. Agregasi fitur User dan Merchant
    user_agg = df.groupby('User').agg(
        user_avg_amount=('Amount', 'mean'),
        user_tx_count=('Amount', 'count')
    ).reset_index()
    
    merchant_agg = df.groupby('Merchant Name').agg(
        merchant_avg_amount=('Amount', 'mean'),
        merchant_tx_count=('Amount', 'count')
    ).reset_index()
    
    # Pastikan urutan sesuai dengan user_mapping
    user_agg['mapped_id'] = user_agg['User'].map(user_mapping)
    user_agg = user_agg.sort_values('mapped_id')
    
    scaler_user = StandardScaler()
    user_feats_scaled = scaler_user.fit_transform(user_agg[['user_avg_amount', 'user_tx_count']])
    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    
    # Pastikan urutan sesuai dengan merchant_mapping
    merchant_agg['mapped_id'] = merchant_agg['Merchant Name'].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values('mapped_id')
    
    scaler_merchant = StandardScaler()
    merchant_feats_scaled = scaler_merchant.fit_transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)
    
    data['user'].x = user_features
    data['merchant'].x = merchant_features
    
    # 3. Fitur dan label Transaction
    x_tx = torch.tensor(df[feature_cols].values, dtype=torch.float)
    y_tx = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    data['transaction'].x = x_tx
    data['transaction'].y = y_tx
    data['transaction'].time = torch.arange(num_transactions)
    
    # 4. Konstruksi Edges (Bipartit / Tripartit dua arah)
    user_src = [user_mapping[uid] for uid in df['User']]
    tx_dst = list(range(num_transactions))
    edge_index_user_tx = torch.tensor([user_src, tx_dst], dtype=torch.long)
    
    merchant_src = [merchant_mapping[mid] for mid in df['Merchant Name']]
    edge_index_merchant_tx = torch.tensor([merchant_src, tx_dst], dtype=torch.long)
    
    data['user', 'buys', 'transaction'].edge_index = edge_index_user_tx
    data['merchant', 'sells', 'transaction'].edge_index = edge_index_merchant_tx
    
    # Reverse edges untuk propagasi pesan dua arah pada GraphSAGE
    data['transaction', 'rev_buys', 'user'].edge_index = edge_index_user_tx.flip([0])
    data['transaction', 'rev_sells', 'merchant'].edge_index = edge_index_merchant_tx.flip([0])
    
    print(f"Graf selesai dibangun: {num_users} users, {num_merchants} merchants, {num_transactions} transaksi.")
    return data



In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self, 
        hidden_channels: int, 
        out_channels: int, 
        num_layers: int, 
        data_metadata: tuple, 
        feature_dims: dict, 
        aggr: str = 'mean',
        dropout: float = 0.2
    ):
        super().__init__()
        
        self.num_layers = num_layers
        self.aggr = aggr
        self.dropout_rate = dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr=aggr
                )
                for edge_type in data_metadata[1]
            })
            self.convs.append(conv)
            
            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier linear bawaan untuk pure end-to-end binary classification
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict, return_embeddings=False):
        # 1. Proyeksi fitur awal
        h_dict = {}
        
        # Node yang memiliki fitur (transaction, user, merchant)
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
                
        # Node tanpa fitur jika ada
        for node_type, param in self.param_embeddings.items():
            num_nodes = num_nodes_dict[node_type]
            h_dict[node_type] = param.expand(num_nodes, -1)
                
        # 2. Message Passing
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            
            # ReLU, BatchNorm, dan Dropout pada transaction nodes
            for node_type in h_dict.keys():
                h_dict[node_type] = F.relu(h_dict[node_type])
                if node_type == 'transaction':
                    h_dict[node_type] = self.bns[i](h_dict[node_type])
                    h_dict[node_type] = self.dropout(h_dict[node_type])
        
        if return_embeddings:
            return h_dict['transaction']
            
        # 3. Klasifikasi biner untuk transaction nodes
        out = self.classifier(h_dict['transaction'])
        return torch.sigmoid(out).squeeze(-1)

def manual_weighted_bce(probs, targets, pos_weight, label_smoothing: float = 0.0):
    """
    Class-weighted Binary Cross-Entropy Loss untuk output probabilitas sigmoid.
    pos_weight = (num_negatives / (num_positives + 1e-7)) atau versi yang diredam (sqrt/clipped).
    Catatan: label_smoothing dinonaktifkan (0.0) pada v5 & exp23 untuk mencegah fatal mathematical
    collision dengan pos_weight yang memicu mode collapse & loss stagnation.
    """
    if label_smoothing > 0.0:
        targets_smooth = targets * (1.0 - label_smoothing) + 0.5 * label_smoothing
    else:
        targets_smooth = targets
        
    loss = - (pos_weight * targets_smooth * torch.log(probs + 1e-7) + 
              (1.0 - targets_smooth) * torch.log(1.0 - probs + 1e-7))
    return loss.mean()

def manual_unweighted_bce(probs, targets):
    """
    Standar Binary Cross-Entropy Loss tanpa pembobotan kelas (untuk monitoring kurva murni).
    """
    loss = - (targets * torch.log(probs + 1e-7) + 
              (1.0 - targets) * torch.log(1.0 - probs + 1e-7))
    return loss.mean()

def focal_loss(probs, targets, alpha: float = 0.25, gamma: float = 2.0):
    """
    Focal Loss untuk menangani class imbalance ekstrim tanpa memicu exploding loss.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    p = probs + 1e-7
    p_t = targets * p + (1.0 - targets) * (1.0 - p)
    alpha_t = targets * alpha + (1.0 - targets) * (1.0 - alpha)
    loss = - alpha_t * ((1.0 - p_t) ** gamma) * torch.log(p_t)
    return loss.mean()



In [ ]:
%%writefile samplers.py
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

def calculate_degree_centrality(data: HeteroData, edge_type: tuple):
    """Menghitung degree centrality sederhana untuk edge type tertentu."""
    src, rel, dst = edge_type
    edge_index = data[edge_type].edge_index
    num_nodes = data[src].num_nodes
    degree = torch.zeros(num_nodes, dtype=torch.float)
    degree.scatter_add_(0, edge_index[0], torch.ones(edge_index.shape[1]))
    return degree

def apply_uniform_sampling(data: HeteroData):
    """
    Baseline: Uniform Random Sampling.
    Memberikan probabilitas yang sama (1.0) untuk semua edge.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.size(1)
        data[edge_type].edge_weight = torch.ones(num_edges, dtype=torch.float)
    return data

def apply_topology_aware_sampling(data: HeteroData):
    """
    Topology-Aware Sampling (Inverse Log-Degree Centrality - v4/v5 Stabilized).
    Menggunakan inverse sublinear log1p(degree) untuk memprioritaskan edge struktural
    tanpa disparitas bobot yang ekstrem.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree)
        log_deg = torch.log1p(deg_src[src_nodes])
        weights = 1.0 / (log_deg + 1e-5)
        min_val = weights.min()
        max_val = weights.max()
        if (max_val - min_val) > 1e-5:
            weights = (weights - min_val) / (max_val - min_val)
        else:
            weights = torch.zeros_like(weights)
        data[edge_type].edge_weight = weights + 0.1
    return data

def apply_importance_based_sampling(data: HeteroData):
    """
    Importance-Based Sampling (Log-Degree Centrality Proxy - v4/v5 Stabilized).
    Menggunakan transformasi sublinear log1p untuk meredam varians sampling
    akibat super-hub merchant dan mencegah anomali ledakan False Positive di akhir training.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree) meredam lonjakan super-hubs
        log_deg = torch.log1p(deg_src[src_nodes])
        min_val = log_deg.min()
        max_val = log_deg.max()
        if (max_val - min_val) > 1e-5:
            importance_weights = (log_deg - min_val) / (max_val - min_val)
        else:
            importance_weights = torch.zeros_like(log_deg)
        data[edge_type].edge_weight = importance_weights + 0.1
    return data

def get_neighbor_loader(data: HeteroData, batch_size: int = 1024, num_neighbors: list = [15, 10], method: str = 'uniform', shuffle: bool = True, input_nodes=None, num_workers: int = 0):
    """
    Mengembalikan PyG NeighborLoader berdasarkan strategi neighbor sampling yang dipilih.
    """
    if method == 'uniform':
        data = apply_uniform_sampling(data)
    elif method == 'topology':
        data = apply_topology_aware_sampling(data)
    elif method == 'importance':
        data = apply_importance_based_sampling(data)
    else:
        raise ValueError(f"Metode sampling tidak dikenal: {method}")

    if input_nodes is None:
        input_nodes = ('transaction', torch.arange(data['transaction'].num_nodes))

    loader = NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        batch_size=batch_size,
        input_nodes=input_nodes,
        shuffle=shuffle,
        weight_attr='edge_weight',
        num_workers=num_workers
    )
    return loader



In [ ]:
%%writefile visualize_results.py
import argparse
import json
import os
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Headless rendering for server/CLI
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from sklearn.metrics import precision_recall_curve, roc_curve, f1_score, precision_score, recall_score

# Set font & style standard
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.autolayout'] = False

# Palette warna harmonis dan profesional (Academic / Modern Analytics)
COLOR_UNIFORM = "#3B82F6"      # Royal Blue
COLOR_TOPOLOGY = "#10B981"     # Emerald Green
COLOR_IMPORTANCE = "#F59E0B"   # Amber Orange
METHOD_COLORS = {
    'uniform': COLOR_UNIFORM,
    'topology': COLOR_TOPOLOGY,
    'importance': COLOR_IMPORTANCE
}
METHOD_NAMES = {
    'uniform': 'Uniform (Baseline)',
    'topology': 'Topology-Aware',
    'importance': 'Importance-Based'
}

def load_data(result_dir: Path):
    """Memuat seluruh data hasil eksperimen EXP23 dan statistik window."""
    methods = ['uniform', 'topology', 'importance']
    results = {}
    for m in methods:
        p = result_dir / f"exp23_{m}_results.json"
        if not p.exists():
            raise FileNotFoundError(f"File hasil tidak ditemukan: {p}")
        with open(p, 'r') as f:
            results[m] = json.load(f)
            
    stat_path = result_dir / "window_statistics.json"
    window_stats = None
    if stat_path.exists():
        with open(stat_path, 'r') as f:
            window_stats = json.load(f)
            
    return results, window_stats

def plot_chart1_primary_metrics(results, out_dir: Path):
    """
    Chart 1: Grouped Bar Chart - Perbandingan Metrik Utama (F1, Recall, Precision, AUCPR, ROC-AUC)
    """
    fig, ax = plt.subplots(figsize=(10.5, 6), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    metrics = ['f1_score', 'recall', 'precision', 'auprc', 'roc_auc']
    metric_labels = ['F1-Score', 'Recall', 'Precision', 'AUCPR (AUPRC)', 'ROC-AUC']
    methods = ['uniform', 'topology', 'importance']
    
    x = np.arange(len(metric_labels))
    width = 0.26
    
    for i, m in enumerate(methods):
        opt = results[m]['metrics_optimal_threshold']
        ind = results[m]['metrics_threshold_independent']
        vals = [
            opt['f1_score'],
            opt['recall'],
            opt['precision'],
            ind['auprc'],
            ind['roc_auc']
        ]
        offset = (i - 1) * width
        rects = ax.bar(x + offset, vals, width, label=METHOD_NAMES[m], 
                       color=METHOD_COLORS[m], edgecolor='#1E293B', linewidth=0.8, alpha=0.92)
        
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.4f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 4),
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=8.5, fontweight='bold',
                        color='#1E293B', rotation=0)

    ax.set_title("EXP23: Komparasi Metrik Klasifikasi Utama antar Strategi Sampling\n"
                 "(Pure End-to-End HeteroGraphSAGE pada Test Set 60 Hari, 283.529 Transaksi)",
                 fontsize=13, fontweight='bold', pad=15, color='#0F172A')
    ax.set_ylabel("Nilai Metrik (0.0 - 1.0)", fontsize=11, fontweight='semibold', labelpad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, fontsize=10.5, fontweight='semibold')
    ax.set_ylim(0.0, 1.05)
    ax.grid(axis='y', linestyle='--', alpha=0.5, color='#CBD5E1')
    ax.legend(loc='lower right', framealpha=0.95, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=10)
    
    plt.tight_layout()
    out_path = out_dir / "chart1_primary_metrics_comparison.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart2_threshold_impact(results, out_dir: Path):
    """
    Chart 2: Dumbbell / Paired Bar Chart - Pengaruh Dynamic Threshold (tau*) vs Default (0.5)
    """
    fig, axes = plt.subplots(1, 3, figsize=(14, 5.2), dpi=300, sharey=True)
    fig.patch.set_facecolor('#FFFFFF')
    
    methods = ['uniform', 'topology', 'importance']
    metrics = ['f1_score', 'recall', 'precision']
    labels = ['F1-Score', 'Recall', 'Precision']
    
    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        def_vals = [results[m]['metrics_default_threshold_05'][k] for k in metrics]
        opt_vals = [results[m]['metrics_optimal_threshold'][k] for k in metrics]
        tau_val = results[m]['threshold_calibration']['optimal_tau']
        
        y = np.arange(len(metrics))
        bar_height = 0.35
        
        r1 = ax.barh(y + bar_height/2, def_vals, bar_height, label='Default (tau = 0.50)', 
                     color='#94A3B8', edgecolor='#334155', linewidth=0.6)
        r2 = ax.barh(y - bar_height/2, opt_vals, bar_height, label=f'Optimal (tau* = {tau_val:.3f})', 
                     color=METHOD_COLORS[m], edgecolor='#0F172A', linewidth=0.8)
        
        for r in r1:
            w = r.get_width()
            ax.annotate(f'{w:.3f}', xy=(w, r.get_y() + r.get_height()/2),
                        xytext=(4, 0), textcoords="offset points", va='center', fontsize=8, color='#475569')
        for r in r2:
            w = r.get_width()
            ax.annotate(f'{w:.3f}', xy=(w, r.get_y() + r.get_height()/2),
                        xytext=(4, 0), textcoords="offset points", va='center', fontsize=8, fontweight='bold', color='#0F172A')

        ax.set_title(f"{METHOD_NAMES[m]}", fontsize=11.5, fontweight='bold', color='#1E293B', pad=10)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=10, fontweight='semibold')
        ax.set_xlim(0, 1.15)
        ax.grid(axis='x', linestyle='--', alpha=0.5, color='#CBD5E1')
        if idx == 0:
            ax.legend(loc='lower right', framealpha=0.95, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=9)

    fig.suptitle("EXP23: Dampak Kalibrasi Threshold Dinamis (tau*) vs Baseline (tau = 0.50)\n"
                 "(Pure GraphSAGE Tanpa XGBoost)",
                 fontsize=13, fontweight='bold', y=1.02, color='#0F172A')
    plt.tight_layout()
    out_path = out_dir / "chart2_threshold_calibration_impact.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart3_confusion_matrix(results, out_dir: Path):
    """
    Chart 3: Confusion Matrix Komparatif (Uniform, Topology, Importance)
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    
    methods = ['uniform', 'topology', 'importance']
    total_samples = 283529
    total_fraud = 421
    total_normal = total_samples - total_fraud
    
    for idx, m in enumerate(methods):
        ax = axes[idx]
        opt = results[m]['metrics_optimal_threshold']
        rec = opt['recall']
        prec = opt['precision']
        
        tp = int(round(rec * total_fraud))
        fn = total_fraud - tp
        fp = int(round((tp / prec) - tp)) if prec > 0 else 0
        tn = total_normal - fp
        
        cm = np.array([[tn, fp], [fn, tp]])
        im = ax.imshow(cm, interpolation='nearest', cmap='Blues', alpha=0.85)
        
        for i in range(2):
            for j in range(2):
                val = cm[i, j]
                color = 'white' if val > (total_normal / 2) else '#0F172A'
                ax.text(j, i, f"{val:,}", ha='center', va='center',
                        color=color, fontsize=11, fontweight='bold')
                
        ax.set_title(f"{METHOD_NAMES[m]}\n(tau* = {opt['threshold']:.3f})", 
                     fontsize=11.5, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Pred Non-Fraud', 'Pred Fraud'], fontsize=9.5, fontweight='semibold')
        ax.set_yticklabels(['True Non-Fraud', 'True Fraud'], fontsize=9.5, fontweight='semibold')
        
    fig.suptitle("EXP23: Heatmap Matriks Kebingungan (Confusion Matrix) Test Set @ tau*\n"
                 "(Pure End-to-End HeteroGraphSAGE Classifier)",
                 fontsize=13, fontweight='bold', y=1.05, color='#0F172A')
    plt.tight_layout()
    out_path = out_dir / "chart3_confusion_matrix_heatmap.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart4_banking_operational(results, out_dir: Path):
    """
    Chart 4: Banking Operational Metric - TP@500 & Precision@500
    """
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    methods = ['uniform', 'topology', 'importance']
    tp_vals = [results[m]['metrics_threshold_independent']['tp_at_500'] for m in methods]
    prec_vals = [results[m]['metrics_threshold_independent']['precision_at_500'] * 100 for m in methods]
    
    x = np.arange(len(methods))
    width = 0.45
    
    rects = ax.bar(x, tp_vals, width, color=[METHOD_COLORS[m] for m in methods], 
                   edgecolor='#0F172A', linewidth=0.8, alpha=0.9)
    
    for idx, rect in enumerate(rects):
        h = rect.get_height()
        p = prec_vals[idx]
        ax.annotate(f"{h} Fraud / 500 Transaksi\n(Precision: {p:.1f}%)",
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#0F172A')

    ax.set_title("EXP23: Efektivitas Alert Triage Harian Perbankan (TP@500 & Precision@500)\n"
                 "(Deteksi Fraud Nyata dalam 500 Transaksi Paling Mencurigakan dari GNN)",
                 fontsize=12, fontweight='bold', pad=15, color='#0F172A')
    ax.set_ylabel("True Positive Fraud (Unit)", fontsize=10.5, fontweight='semibold')
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_NAMES[m] for m in methods], fontsize=10.5, fontweight='semibold')
    ax.set_ylim(0, 500)
    ax.grid(axis='y', linestyle='--', alpha=0.5, color='#CBD5E1')
    
    plt.tight_layout()
    out_path = out_dir / "chart4_banking_tp_at_500.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart5_lift_comparison(results, out_dir: Path):
    """
    Chart 5: Peningkatan Deteksi Fraud Top-1% (Lift@1%)
    """
    fig, ax = plt.subplots(figsize=(7.5, 4.8), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    methods = ['uniform', 'topology', 'importance']
    lifts = [results[m]['metrics_threshold_independent']['lift_1_percent'] for m in methods]
    
    bars = ax.barh([METHOD_NAMES[m] for m in methods], lifts, height=0.45,
                   color=[METHOD_COLORS[m] for m in methods], edgecolor='#0F172A', linewidth=0.8)
    
    for b in bars:
        w = b.get_width()
        ax.annotate(f"{w:.2f}x Lipat",
                    xy=(w, b.get_y() + b.get_height() / 2),
                    xytext=(6, 0), textcoords="offset points",
                    va='center', fontsize=10, fontweight='bold', color='#0F172A')

    ax.set_title("EXP23: Rasio Peningkatan Deteksi Fraud (Lift @ Top-1% Transaksi)\n"
                 "(Dibandingkan Tingkat Fraud Acak Alami Base Rate)",
                 fontsize=12, fontweight='bold', pad=12, color='#0F172A')
    ax.set_xlabel("Nilai Lift (Kelipatan Keunggulan terhadap Base Rate)", fontsize=10, fontweight='semibold')
    ax.set_xlim(0, max(lifts) * 1.25 if max(lifts) > 0 else 100)
    ax.grid(axis='x', linestyle='--', alpha=0.5, color='#CBD5E1')
    
    plt.tight_layout()
    out_path = out_dir / "chart5_lift_at_1_percent.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart6_runtime_breakdown(results, out_dir: Path):
    """
    Chart 6: Runtime Breakdown Komputasi Pipeline Pure GNN
    """
    fig, ax = plt.subplots(figsize=(9, 5), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    methods = ['uniform', 'topology', 'importance']
    rt_data = {
        'Preprocessing': [results[m]['runtime_seconds']['data_preprocessing'] for m in methods],
        'GNN Training (10 Ep)': [results[m]['runtime_seconds']['gnn_training'] for m in methods],
        'GNN Val Tuning': [results[m]['runtime_seconds']['gnn_val_tuning'] for m in methods],
        'GNN Test Inference': [results[m]['runtime_seconds']['gnn_test_inference'] for m in methods]
    }
    
    bottom = np.zeros(len(methods))
    colors = ['#94A3B8', '#3B82F6', '#8B5CF6', '#10B981']
    
    for idx, (stage, times) in enumerate(rt_data.items()):
        bars = ax.bar([METHOD_NAMES[m] for m in methods], times, bottom=bottom, label=stage,
                      color=colors[idx], edgecolor='#1E293B', linewidth=0.7, width=0.45)
        bottom += np.array(times)
        
    for i, total_sec in enumerate(bottom):
        ax.annotate(f"Total: {total_sec:.1f}s\n({total_sec/60:.1f}m)",
                    xy=(i, total_sec), xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='bold', color='#0F172A')

    ax.set_title("EXP23: Waktu Eksekusi Pipeline Komputasi Pure GNN (Detik)\n"
                 "(Bebas Overhead Ekstraksi Fitur & Pelatihan XGBoost)",
                 fontsize=12, fontweight='bold', pad=15, color='#0F172A')
    ax.set_ylabel("Waktu Komputasi (Detik)", fontsize=10.5, fontweight='semibold')
    ax.set_ylim(0, max(bottom) * 1.25)
    ax.grid(axis='y', linestyle='--', alpha=0.5, color='#CBD5E1')
    ax.legend(loc='upper right', framealpha=0.95, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=9)
    
    plt.tight_layout()
    out_path = out_dir / "chart6_runtime_breakdown.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart7_timeline(stats, out_dir: Path):
    """
    Chart 7: Lini Masa Desain Temporal Rolling Window (SHINE FRAUD II Protocol)
    """
    if not stats:
        return
    fig, ax = plt.subplots(figsize=(12, 3.8), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    splits = stats['splits']
    order = ['train', 'validation', 'embargo_gap', 'test']
    split_colors = {
        'train': '#3B82F6',
        'validation': '#F59E0B',
        'embargo_gap': '#EF4444',
        'test': '#10B981'
    }
    
    for idx, key in enumerate(order):
        sp = splits[key]
        start_d = datetime.strptime(sp['start_date'][:10], '%Y-%m-%d')
        end_d = datetime.strptime(sp['end_date'][:10], '%Y-%m-%d')
        days = (end_d - start_d).days
        
        ax.barh(0, days, left=start_d, height=0.5, color=split_colors[key],
                edgecolor='#0F172A', linewidth=0.8, alpha=0.9)
        
        mid_date = start_d + timedelta(days=days/2)
        ax.text(mid_date, 0, f"{sp['name']}\n{days} Hari\n({sp['num_transactions']:,} tx)",
                ha='center', va='center', fontsize=8.5, fontweight='bold', color='#FFFFFF')

    ax.set_yticks([])
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.set_title("EXP23: Desain Protokol Jendela Waktu Bergulir (SHINE FRAUD II Alignment)\n"
                 "Train: 300 Hari | Validation: 30 Hari | Embargo: 8 Hari | Test: 60 Hari",
                 fontsize=12, fontweight='bold', pad=12, color='#0F172A')
    ax.grid(axis='x', linestyle='--', alpha=0.5, color='#CBD5E1')
    
    plt.tight_layout()
    out_path = out_dir / "chart7_temporal_window_timeline.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart8_partition_characteristics(stats, out_dir: Path):
    """
    Chart 8: Karakteristik Volume Transaksi & Rasio Fraud Tiap Partisi
    """
    if not stats:
        return
    fig, ax1 = plt.subplots(figsize=(9, 4.8), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax1.set_facecolor('#F8FAFC')
    
    splits = stats['splits']
    order = ['train', 'validation', 'embargo_gap', 'test']
    names = [splits[k]['name'] for k in order]
    tx_counts = [splits[k]['num_transactions'] for k in order]
    fraud_rates = [splits[k]['fraud_rate_pct'] for k in order]
    
    x = np.arange(len(order))
    width = 0.4
    
    rects = ax1.bar(x, tx_counts, width, color='#3B82F6', edgecolor='#1E293B', alpha=0.85, label='Volume Transaksi')
    ax1.set_ylabel("Volume Transaksi", fontsize=10.5, fontweight='semibold', color='#1E293B')
    ax1.set_xticks(x)
    ax1.set_xticklabels(names, fontsize=10, fontweight='semibold')
    
    ax2 = ax1.twinx()
    line = ax2.plot(x, fraud_rates, color='#DC2626', marker='o', linewidth=2.4, markersize=7, label='Tingkat Fraud (%)')
    ax2.set_ylabel("Tingkat Fraud (%)", fontsize=10.5, fontweight='semibold', color='#DC2626')
    ax2.set_ylim(0, max(fraud_rates) * 1.5 if max(fraud_rates) > 0 else 1.0)
    
    for rect in rects:
        h = rect.get_height()
        ax1.annotate(f"{h:,}", xy=(rect.get_x() + rect.get_width()/2, h),
                     xytext=(0, 4), textcoords="offset points", ha='center', fontsize=8.5, fontweight='bold')
    for i, fr in enumerate(fraud_rates):
        ax2.annotate(f"{fr:.3f}%", xy=(x[i], fr), xytext=(0, 6), textcoords="offset points",
                     ha='center', fontsize=8.5, fontweight='bold', color='#DC2626')

    ax1.set_title("EXP23: Karakteristik Volume Transaksi dan Rasio Fraud per Partisi Temporal",
                  fontsize=12, fontweight='bold', pad=12, color='#0F172A')
    plt.tight_layout()
    out_path = out_dir / "chart8_partition_characteristics.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart9_radar_synthesis(results, out_dir: Path):
    """
    Chart 9: Radar Chart Sintesis Multi-Dimensi (7 Dimensi Evaluasi)
    """
    categories = ['F1-Score', 'Recall', 'Precision', 'AUCPR', 'ROC-AUC', 'Lift@1% (Norm)', 'TP@500 (Norm)']
    N = len(categories)
    
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(7.5, 7.5), subplot_kw=dict(polar=True), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    methods = ['uniform', 'topology', 'importance']
    max_lift = max([results[m]['metrics_threshold_independent']['lift_1_percent'] for m in methods] or [1.0])
    
    for m in methods:
        opt = results[m]['metrics_optimal_threshold']
        ind = results[m]['metrics_threshold_independent']
        
        values = [
            opt['f1_score'],
            opt['recall'],
            opt['precision'],
            ind['auprc'],
            ind['roc_auc'],
            ind['lift_1_percent'] / max_lift if max_lift > 0 else 0.0,
            ind['tp_at_500'] / 500.0
        ]
        values += values[:1]
        
        ax.plot(angles, values, linewidth=2, linestyle='solid', label=METHOD_NAMES[m], color=METHOD_COLORS[m])
        ax.fill(angles, values, color=METHOD_COLORS[m], alpha=0.15)
        
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=9.5, fontweight='semibold')
    ax.set_ylim(0, 1.05)
    ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1), fontsize=9, framealpha=0.95)
    ax.set_title("EXP23: Radar Sintesis Evaluasi Multi-Dimensi (Pure GraphSAGE)",
                 fontsize=12, fontweight='bold', pad=20, color='#0F172A')
    
    plt.tight_layout()
    out_path = out_dir / "chart9_radar_multicriteria_synthesis.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_continuous_curves(predictions_dict, results, out_dir: Path):
    """
    Chart 10: Precision-Recall Curve (PR-Curve)
    Chart 11: ROC Curve (Receiver Operating Characteristic)
    """
    methods = ['uniform', 'topology', 'importance']
    
    # 1. PR Curve
    fig, ax = plt.subplots(figsize=(8.5, 6), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    for m in methods:
        if m not in predictions_dict:
            continue
        y_true = predictions_dict[m]['test_labels']
        y_scores = predictions_dict[m]['test_probs']
        prec, rec, _ = precision_recall_curve(y_true, y_scores)
        auprc_val = results[m]['metrics_threshold_independent']['auprc']
        ax.plot(rec, prec, color=METHOD_COLORS[m], linewidth=2.2, label=f"{METHOD_NAMES[m]} (AUPRC = {auprc_val:.4f})")
        
        # Titik operasi optimal tau*
        opt_prec = results[m]['metrics_optimal_threshold']['precision']
        opt_rec = results[m]['metrics_optimal_threshold']['recall']
        ax.scatter(opt_rec, opt_prec, color=METHOD_COLORS[m], s=70, edgecolor='#0F172A', zorder=5)

    base_rate = 421 / 283529
    ax.axhline(base_rate, color='#94A3B8', linestyle=':', label=f'Baseline Fraud Rate ({base_rate*100:.3f}%)')
    ax.set_title("EXP23: Precision-Recall Curve Kontinu pada Test Set (Pure GraphSAGE)", fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel("Recall (Sensitivitas)", fontsize=10.5, fontweight='semibold')
    ax.set_ylabel("Precision (PPV)", fontsize=10.5, fontweight='semibold')
    ax.set_ylim(-0.02, 1.05)
    ax.set_xlim(-0.02, 1.02)
    ax.grid(linestyle='--', alpha=0.5, color='#CBD5E1')
    ax.legend(loc='lower left', fontsize=9.5, framealpha=0.95)
    plt.tight_layout()
    pr_path = out_dir / "chart10_precision_recall_curves.png"
    plt.savefig(pr_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {pr_path.name}")

    # 2. ROC Curve
    fig, ax = plt.subplots(figsize=(8.5, 6), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    for m in methods:
        if m not in predictions_dict:
            continue
        y_true = predictions_dict[m]['test_labels']
        y_scores = predictions_dict[m]['test_probs']
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_val = results[m]['metrics_threshold_independent']['roc_auc']
        ax.plot(fpr, tpr, color=METHOD_COLORS[m], linewidth=2.2, label=f"{METHOD_NAMES[m]} (ROC-AUC = {roc_val:.4f})")

    ax.plot([0, 1], [0, 1], color='#94A3B8', linestyle='--', label='Random Guess (AUC = 0.5000)')
    ax.set_title("EXP23: ROC Curve Kontinu pada Test Set (Pure GraphSAGE)", fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel("False Positive Rate (1 - Spesifisitas)", fontsize=10.5, fontweight='semibold')
    ax.set_ylabel("True Positive Rate (Recall)", fontsize=10.5, fontweight='semibold')
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlim(-0.02, 1.02)
    ax.grid(linestyle='--', alpha=0.5, color='#CBD5E1')
    ax.legend(loc='lower right', fontsize=9.5, framealpha=0.95)
    plt.tight_layout()
    roc_path = out_dir / "chart11_roc_curves.png"
    plt.savefig(roc_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {roc_path.name}")

def plot_epoch_learning_curves(histories, out_dir: Path):
    """
    Chart 12: Kurva Pembelajaran Loss GNN (Train vs Val)
    Chart 13: Kurva Pembelajaran Akurasi GNN
    Chart 14: Kurva Dinamika Multi-Metrik Validation per Epoch
    """
    methods = ['uniform', 'topology', 'importance']
    
    # Chart 12: Loss Curves
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), dpi=300, sharey=True)
    fig.patch.set_facecolor('#FFFFFF')
    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        hist = histories[m]['gnn_history']
        epochs = hist['epochs']
        ax.plot(epochs, hist['train_loss'], marker='o', linewidth=2.4, color=METHOD_COLORS[m], label='Train Weighted Loss')
        ax.plot(epochs, hist['val_loss'], marker='^', linewidth=2.2, linestyle='--', color='#DC2626', label='Val Weighted Loss')
        if 'train_loss_unweighted' in hist:
            ax.plot(epochs, hist['train_loss_unweighted'], linewidth=1.5, linestyle=':', color='#64748B', label='Train Unweighted Loss')
        ax.set_title(METHOD_NAMES[m], fontsize=11.5, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Epoch", fontsize=10, fontweight='semibold')
        if idx == 0:
            ax.set_ylabel("Binary Cross-Entropy Loss", fontsize=10.5, fontweight='semibold')
        ax.set_xticks(epochs)
        ax.grid(linestyle='--', alpha=0.5, color='#CBD5E1')
        ax.legend(loc='upper right', fontsize=8.5, framealpha=0.95)
    fig.suptitle("EXP23: Kurva Pembelajaran Loss GNN Tiap Epoch (Train vs Val)", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    out12 = out_dir / "chart12_learning_curves_loss.png"
    plt.savefig(out12, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out12.name}")

    # Chart 13: Accuracy Curves
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), dpi=300, sharey=True)
    fig.patch.set_facecolor('#FFFFFF')
    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        hist = histories[m]['gnn_history']
        epochs = hist['epochs']
        ax.plot(epochs, [a * 100 for a in hist['train_accuracy']], marker='o', linewidth=2.4, color=METHOD_COLORS[m], label='Train Acc (%)')
        ax.plot(epochs, [a * 100 for a in hist['val_accuracy']], marker='^', linewidth=2.2, linestyle='--', color='#DC2626', label='Val Acc @ 0.5 (%)')
        if 'val_calibrated_accuracy' in hist:
            ax.plot(epochs, [a * 100 for a in hist['val_calibrated_accuracy']], marker='d', linewidth=2.0, linestyle='-.', color='#059669', label='Val Acc @ tau* (%)')
        ax.set_title(METHOD_NAMES[m], fontsize=11.5, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Epoch", fontsize=10, fontweight='semibold')
        if idx == 0:
            ax.set_ylabel("Akurasi (%)", fontsize=10.5, fontweight='semibold')
        ax.set_xticks(epochs)
        ax.grid(linestyle='--', alpha=0.5, color='#CBD5E1')
        ax.legend(loc='lower right', fontsize=8.5, framealpha=0.95)
    fig.suptitle("EXP23: Kurva Pembelajaran Akurasi GNN Tiap Epoch", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    out13 = out_dir / "chart13_learning_curves_accuracy.png"
    plt.savefig(out13, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out13.name}")

    # Chart 14: Multi-Metric Dynamics on Validation
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    metrics_to_plot = [
        ('val_f1', 'F1-Score (Validation)', axes[0, 0]),
        ('val_recall', 'Recall / Hit Rate (Validation)', axes[0, 1]),
        ('val_precision', 'Precision / PPV (Validation)', axes[1, 0]),
        ('val_auprc', 'AUCPR / Average Precision (Validation)', axes[1, 1])
    ]
    for key, title, ax in metrics_to_plot:
        ax.set_facecolor('#F8FAFC')
        for m in methods:
            hist = histories[m]['gnn_history']
            epochs = hist['epochs']
            ax.plot(epochs, hist[key], marker='o', linewidth=2.2, markersize=5, color=METHOD_COLORS[m], label=METHOD_NAMES[m])
        ax.set_title(title, fontsize=11.5, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Epoch", fontsize=10, fontweight='semibold')
        ax.set_xticks(epochs)
        ax.grid(linestyle='--', alpha=0.5, color='#CBD5E1')
        ax.legend(loc='best', fontsize=9, framealpha=0.95)
    fig.suptitle("EXP23: Dinamika Multi-Metrik GNN pada Validation Set per Epoch", fontsize=13.5, fontweight='bold', y=1.02)
    plt.tight_layout()
    out14 = out_dir / "chart14_validation_multimetric_dynamics.png"
    plt.savefig(out14, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out14.name}")

def plot_chart15_threshold_sensitivity_curves(predictions_dict, results, out_dir: Path):
    """
    Chart 15: Kurva Sensitivitas Kalibrasi Threshold Dinamis GNN pada Test Set
    Menggantikan kurva XGBoost dengan analisis trade-off Precision-Recall-F1 sepanjang tau in [0.01, 0.50].
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), dpi=300, sharey=True)
    fig.patch.set_facecolor('#FFFFFF')
    methods = ['uniform', 'topology', 'importance']
    thresholds = np.linspace(0.01, 0.50, 100)
    
    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        if m not in predictions_dict:
            continue
        probs = predictions_dict[m]['test_probs']
        labels = predictions_dict[m]['test_labels']
        best_tau = results[m]['threshold_calibration']['optimal_tau']
        
        f1_list, prec_list, rec_list = [], [], []
        for t in thresholds:
            preds = (probs >= t).astype(int)
            f1_list.append(f1_score(labels, preds, zero_division=0))
            prec_list.append(precision_score(labels, preds, zero_division=0))
            rec_list.append(recall_score(labels, preds, zero_division=0))
            
        ax.plot(thresholds, f1_list, linewidth=2.4, color='#2563EB', label='F1-Score')
        ax.plot(thresholds, prec_list, linewidth=2.0, color='#059669', linestyle='--', label='Precision')
        ax.plot(thresholds, rec_list, linewidth=2.0, color='#DC2626', linestyle=':', label='Recall')
        ax.axvline(x=best_tau, color='#D97706', linestyle='-.', linewidth=2.0, label=f'Optimal $\\tau^* = {best_tau:.3f}$')
        
        ax.set_title(f"{METHOD_NAMES[m]}", fontsize=12, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Ambang Batas Klasifikasi ($\\tau$)", fontsize=10.5, fontweight='bold', color='#334155')
        if idx == 0:
            ax.set_ylabel("Nilai Metrik (0.0 - 1.0)", fontsize=11, fontweight='bold', color='#334155')
        ax.set_ylim(-0.02, 1.05)
        ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
        ax.legend(loc='lower left', fontsize=9, framealpha=0.95, facecolor='#FFFFFF')
        
    fig.suptitle("EXP23: Kurva Sensitivitas Kalibrasi Threshold Dinamis HeteroGraphSAGE pada Test Set\n"
                 "(Dinamika Trade-off Precision-Recall-F1 sepanjang Ambang Batas Decision)",
                 fontsize=13.5, fontweight='bold', y=1.03, color='#0F172A')
    plt.tight_layout()
    out_path = out_dir / "chart15_gnn_threshold_sensitivity.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def exponential_moving_average(values, alpha=0.02):
    """Menghitung Exponential Moving Average untuk menghaluskan tren kurva."""
    if len(values) == 0:
        return np.array([])
    ema = np.empty_like(values, dtype=np.float64)
    ema[0] = values[0]
    for i in range(1, len(values)):
        ema[i] = alpha * values[i] + (1.0 - alpha) * ema[i - 1]
    return ema

def plot_step_curves(histories, out_dir: Path):
    """
    Chart 16: Kurva Konvergensi Loss GNN per-Step Mini-Batch Iterasi
    Chart 17: Kurva Konvergensi Akurasi GNN per-Step Mini-Batch Iterasi
    """
    methods = ['uniform', 'topology', 'importance']
    has_step = any('gnn_step_history' in histories.get(m, {}) and len(histories[m]['gnn_step_history'].get('steps', [])) > 0 for m in methods)
    if not has_step:
        print("[INFO] Data gnn_step_history belum tersedia untuk Chart 16 & 17. Melewati...")
        return
        
    # Chart 16: Step Loss
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        step_hist = histories.get(m, {}).get('gnn_step_history', {})
        steps = np.array(step_hist.get('steps', []))
        loss = np.array(step_hist.get('train_loss', []))
        epochs_arr = np.array(step_hist.get('epochs', []))
        if len(steps) > 0 and len(loss) > 0:
            ema = exponential_moving_average(loss, alpha=0.02)
            ax.plot(steps, loss, color=METHOD_COLORS[m], alpha=0.18, linewidth=0.8, label='Raw Batch Loss')
            ax.plot(steps, ema, color=METHOD_COLORS[m], linewidth=2.6, label='Smoothed Trend (EMA)')
            if len(epochs_arr) > 0:
                epoch_changes = np.where(np.diff(epochs_arr) > 0)[0]
                for ep_idx in epoch_changes:
                    ax.axvline(x=steps[ep_idx], color='#94A3B8', linestyle=':', alpha=0.5, linewidth=0.9)
        ax.set_title(METHOD_NAMES[m], fontsize=12, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Langkah Pembaruan Mini-Batch (Steps)", fontsize=10.5, fontweight='bold', color='#334155')
        if idx == 0:
            ax.set_ylabel("Loss Terbobot per-Step", fontsize=11, fontweight='bold', color='#334155')
        ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
        ax.legend(loc='upper right', fontsize=9, framealpha=0.95, facecolor='#FFFFFF')
    fig.suptitle("EXP23: Kurva Konvergensi Loss GNN per-Step Mini-Batch Iterasi", fontsize=13.5, fontweight='bold', y=1.03)
    plt.tight_layout()
    out16 = out_dir / "chart16_step_loss_convergence.png"
    plt.savefig(out16, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out16.name}")

    # Chart 17: Step Accuracy
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        step_hist = histories.get(m, {}).get('gnn_step_history', {})
        steps = np.array(step_hist.get('steps', []))
        acc = np.array(step_hist.get('train_accuracy', []))
        epochs_arr = np.array(step_hist.get('epochs', []))
        if len(steps) > 0 and len(acc) > 0:
            ema = exponential_moving_average(acc, alpha=0.02)
            ax.plot(steps, [a * 100 for a in acc], color=METHOD_COLORS[m], alpha=0.18, linewidth=0.8, label='Raw Batch Acc (%)')
            ax.plot(steps, [a * 100 for a in ema], color=METHOD_COLORS[m], linewidth=2.6, label='Smoothed Trend (EMA)')
            if len(epochs_arr) > 0:
                epoch_changes = np.where(np.diff(epochs_arr) > 0)[0]
                for ep_idx in epoch_changes:
                    ax.axvline(x=steps[ep_idx], color='#94A3B8', linestyle=':', alpha=0.5, linewidth=0.9)
        ax.set_title(METHOD_NAMES[m], fontsize=12, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Langkah Pembaruan Mini-Batch (Steps)", fontsize=10.5, fontweight='bold', color='#334155')
        if idx == 0:
            ax.set_ylabel("Akurasi per-Step (%)", fontsize=11, fontweight='bold', color='#334155')
        ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
        ax.legend(loc='lower right', fontsize=9, framealpha=0.95, facecolor='#FFFFFF')
    fig.suptitle("EXP23: Kurva Konvergensi Akurasi GNN per-Step Mini-Batch Iterasi", fontsize=13.5, fontweight='bold', y=1.03)
    plt.tight_layout()
    out17 = out_dir / "chart17_step_accuracy_convergence.png"
    plt.savefig(out17, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out17.name}")

def main():
    parser = argparse.ArgumentParser(description="Visualisasi Hasil Eksperimen 23 v2 (Pure End-to-End HeteroGraphSAGE)")
    parser.add_argument('--result-dir', type=str, default='../../result/exp23/v2', help='Path ke folder direktori hasil EXP23 v2')
    parser.add_argument('--output-dir', type=str, default='../../result/exp23/v2/plots', help='Path folder simpan gambar visualisasi')
    args = parser.parse_args()
    
    res_dir = Path(args.result_dir)
    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n[INFO] Memuat berkas hasil evaluasi dari: {res_dir.resolve()}")
    results, window_stats = load_data(res_dir)
    
    # Muat predictions npz jika tersedia
    methods = ['uniform', 'topology', 'importance']
    predictions_dict = {}
    for m in methods:
        pred_p = res_dir / f"exp23_{m}_predictions.npz"
        if pred_p.exists():
            data = np.load(pred_p)
            predictions_dict[m] = {
                'test_probs': data['test_probs'],
                'test_labels': data['test_labels']
            }
            
    # Muat history json jika tersedia
    histories_dict = {}
    for m in methods:
        hist_p = res_dir / f"exp23_{m}_history.json"
        if hist_p.exists():
            with open(hist_p, 'r') as f:
                histories_dict[m] = json.load(f)
                
    print(f"[INFO] Menghasilkan 16 Visualisasi Ilmiah Resolusi Tinggi (300 DPI) ke {out_dir.resolve()}...")
    plot_chart1_primary_metrics(results, out_dir)
    plot_chart2_threshold_impact(results, out_dir)
    plot_chart3_confusion_matrix(results, out_dir)
    plot_chart4_banking_operational(results, out_dir)
    plot_chart5_lift_comparison(results, out_dir)
    plot_chart6_runtime_breakdown(results, out_dir)
    plot_chart7_timeline(window_stats, out_dir)
    plot_chart8_partition_characteristics(window_stats, out_dir)
    plot_chart9_radar_synthesis(results, out_dir)
    
    if len(predictions_dict) == 3:
        plot_continuous_curves(predictions_dict, results, out_dir)
        plot_chart15_threshold_sensitivity_curves(predictions_dict, results, out_dir)
    else:
        print("[WARNING] Berkas predictions.npz tidak lengkap untuk semua metode. Melewati Chart 10, 11, 15.")
        
    if len(histories_dict) == 3:
        plot_epoch_learning_curves(histories_dict, out_dir)
        plot_step_curves(histories_dict, out_dir)
    else:
        print("[WARNING] Berkas history.json tidak lengkap untuk semua metode. Melewati Chart 12, 13, 14, 16, 17.")
        
    print(f"\n🎉 Seluruh 16 chart visualisasi EXP23 berhasil digenerate di: {out_dir.resolve()} 🎉\n")

if __name__ == '__main__':
    main()



In [ ]:
%%writefile main.py
import argparse
import time
import json
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from sklearn.metrics import (
    f1_score, recall_score, precision_score, 
    accuracy_score, roc_auc_score, average_precision_score, 
    confusion_matrix
)

from config import load_config
from data_preprocessing import load_and_preprocess_data, get_temporal_window_split
from graph_builder import build_hetero_graph, graph_level_undersample
from samplers import get_neighbor_loader
from model import HeteroGraphSAGE, manual_weighted_bce, manual_unweighted_bce

def find_optimal_threshold(val_probs, val_labels, thresholds=None):
    """
    Mencari threshold optimal tau* yang memaksimalkan F1-score pada Validation set.
    """
    if thresholds is None:
        thresholds = np.arange(0.01, 0.51, 0.005)
        
    best_tau = 0.5
    best_f1 = 0.0
    best_metrics = {}
    
    for tau in thresholds:
        preds = (val_probs >= tau).astype(int)
        f1 = f1_score(val_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_tau = float(tau)
            best_metrics = {
                "tau": float(tau),
                "val_f1": float(f1),
                "val_precision": float(precision_score(val_labels, preds, zero_division=0)),
                "val_recall": float(recall_score(val_labels, preds, zero_division=0))
            }
            
    if best_f1 == 0.0:
        best_tau = 0.5
        best_metrics = {
            "tau": 0.5,
            "val_f1": 0.0,
            "val_precision": 0.0,
            "val_recall": 0.0
        }
        
    return best_tau, best_metrics

def predict_gnn(model, loader, device):
    """
    Menjalankan inferensi probabilitas sigmoid murni dari HeteroGraphSAGE.
    Mengembalikan probabilitas kontinu p_i in [0, 1], label ground truth, dan waktu inferensi.
    """
    model.eval()
    all_probs = []
    all_targets = []
    start_time = time.time()
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            batch_size = batch['transaction'].batch_size
            all_probs.append(probs[:batch_size].cpu())
            all_targets.append(batch['transaction'].y[:batch_size].cpu())
            
    inference_time = time.time() - start_time
    probs_cat = torch.cat(all_probs).numpy().ravel()
    targets_cat = torch.cat(all_targets).numpy().ravel().astype(np.int64)
    return probs_cat, targets_cat, inference_time

def evaluate_gnn(model, loader, device, pos_weight):
    """
    Mengevaluasi model GNN pada loader tertentu.
    Menghitung loss terbobot, loss unweighted, metrik @ 0.5, metrik @ tau*, auprc, dan roc_auc.
    """
    model.eval()
    total_loss = 0.0
    total_loss_unw = 0.0
    total_samples = 0
    all_probs = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            loss = manual_weighted_bce(probs, targets, pos_weight)
            loss_unw = manual_unweighted_bce(probs, targets)
            
            total_loss += loss.item() * batch_size_current
            total_loss_unw += loss_unw.item() * batch_size_current
            total_samples += batch_size_current
            all_probs.append(probs.cpu())
            all_targets.append(targets.cpu())
            
    all_probs = torch.cat(all_probs).numpy().ravel()
    all_targets = torch.cat(all_targets).numpy().ravel().astype(np.int64)
    avg_loss = total_loss / max(1, total_samples)
    avg_loss_unw = total_loss_unw / max(1, total_samples)
    
    # 1. Metrik @ Baseline Fixed Threshold 0.5
    preds_05 = (all_probs >= 0.5).astype(int)
    acc_05 = float(accuracy_score(all_targets, preds_05))
    f1_05 = float(f1_score(all_targets, preds_05, zero_division=0))
    prec_05 = float(precision_score(all_targets, preds_05, zero_division=0))
    rec_05 = float(recall_score(all_targets, preds_05, zero_division=0))
    
    # 2. Metrik Bebas Threshold
    aucpr = float(average_precision_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    roc_auc = float(roc_auc_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    
    # 3. Metrik @ Optimal Threshold tau*
    best_tau, tau_stats = find_optimal_threshold(all_probs, all_targets)
    preds_tau = (all_probs >= best_tau).astype(int)
    acc_tau = float(accuracy_score(all_targets, preds_tau))
    
    return {
        'loss': float(avg_loss),
        'loss_unweighted': float(avg_loss_unw),
        'accuracy': acc_05,
        'f1_score': f1_05,
        'precision': prec_05,
        'recall': rec_05,
        'auprc': aucpr,
        'roc_auc': roc_auc,
        'optimal_tau': float(best_tau),
        'calibrated_accuracy': acc_tau,
        'calibrated_f1': float(tau_stats['val_f1']),
        'calibrated_precision': float(tau_stats['val_precision']),
        'calibrated_recall': float(tau_stats['val_recall'])
    }

def main():
    parser = argparse.ArgumentParser(description="Eksperimen 23: Pure End-to-End HeteroGraphSAGE Classification (Tanpa XGBoost)")
    parser.add_argument('--config', type=str, default='config.reference.yaml', help='Path ke file konfigurasi YAML')
    parser.add_argument('--sampling', type=str, choices=['uniform', 'topology', 'importance'], default=None, help='Metode neighbor sampling')
    parser.add_argument('--reference-date', type=str, default=None, help='Override tanggal T0 window (format: YYYY-MM-DD)')
    parser.add_argument('--nrows', type=int, default=None, help='Limit baris data untuk debugging lokal')
    parser.add_argument('--start-year', type=int, default=None, help='Override filter tahun awal dataset')
    args = parser.parse_args()
    
    total_pipeline_start = time.time()
    
    # 1. Load Konfigurasi
    cfg = load_config(args.config)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*75}")
    print(f"  EXP23 v2: PURE END-TO-END HETEROGRAPHSAGE CLASSIFICATION")
    print(f"  (256-dim, [25, 10] Neighbors, Early Stopping Patience=15, No XGBoost)")
    print(f"{'='*75}")
    print(f"Device               : {device}")
    
    sampling_method = args.sampling if args.sampling is not None else cfg['sampling']['method']
    print(f"Sampling Method      : {sampling_method}")
    
    # Override time_window parameters jika disediakan di CLI
    tw_cfg = cfg.get('time_window', {})
    reference_date = args.reference_date if args.reference_date is not None else tw_cfg.get('reference_date', '2018-01-01')
    train_days = tw_cfg.get('train_days', 300)
    val_days = tw_cfg.get('val_days', 30)
    gap_days = tw_cfg.get('gap_days', 8)
    test_days = tw_cfg.get('test_days', 60)
    
    start_year = args.start_year if args.start_year is not None else cfg['data'].get('start_year', 2016)
    dataset_path = cfg['data']['transactions']
    nrows = args.nrows if args.nrows is not None else cfg['data'].get('max_rows', None)
    
    # Siapkan direktori output
    model_dir = Path(cfg.get('output', {}).get('model_dir', '../../model/exp23/v2'))
    result_dir = Path(cfg.get('output', {}).get('result_dir', '../../result/exp23/v2'))
    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    # 2. Preprocessing Data & Filter Temporal 2016-2020
    print("\n--- [Langkah 1: Preprocessing Data (2016-2020)] ---")
    data_prep_start = time.time()
    df, feature_cols = load_and_preprocess_data(dataset_path, start_year=start_year, nrows=nrows)
    data_prep_time = time.time() - data_prep_start
    print(f"Data preprocessing selesai dalam {data_prep_time:.2f} detik. Total baris: {len(df)}")
    
    # 3. Pembagian Temporal Window (300d train, 30d val, 8d gap, 60d test)
    print("\n--- [Langkah 2: Pembagian Jendela Temporal SHINE FRAUD II] ---")
    train_df, val_df, test_df, gap_df, window_stats = get_temporal_window_split(
        df, 
        reference_date=reference_date,
        train_days=train_days,
        val_days=val_days,
        gap_days=gap_days,
        test_days=test_days
    )
    
    print("\nStatistik Partisi Jendela Temporal:")
    for split_key, sp in window_stats['splits'].items():
        print(f"  • {sp['name']:12s}: {sp['num_transactions']:8d} transaksi | "
              f"{sp['num_fraud']:5d} fraud ({sp['fraud_rate_pct']:6.3f}%) | "
              f"Rentang: {sp['start_date'][:10]} s/d {sp['end_date'][:10]}")
              
    # Simpan window_statistics.json
    stats_path = result_dir / "window_statistics.json"
    with open(stats_path, 'w') as f:
        json.dump(window_stats, f, indent=4)
    print(f"Statistik jendela waktu tersimpan ke: {stats_path}")
    
    if len(train_df) == 0 or len(test_df) == 0:
        raise ValueError("Dataframe Train atau Test kosong! Periksa kembali parameter reference_date atau nrows.")

    # 4. Undersampling (Default: 1.0 = Tanpa Undersampling)
    undersample_ratio = cfg['sampling'].get('undersampling_ratio', 1.0)
    train_df_sampled = graph_level_undersample(train_df, ratio=undersample_ratio)

    # 5. Konstruksi Graf Heterogen (PyG)
    print("\n--- [Langkah 3: Membangun Graf Train] ---")
    train_data = build_hetero_graph(train_df_sampled, feature_cols)
    
    batch_size = cfg['training'].get('batch_size', 1024)
    num_neighbors = cfg['training'].get('num_neighbors', [25, 10])
    
    train_loader = get_neighbor_loader(
        train_data, 
        batch_size=batch_size, 
        num_neighbors=num_neighbors,
        method=sampling_method, 
        shuffle=True
    )
    
    # 6. Definisi Arsitektur Model GNN (HeteroGraphSAGE)
    feature_dims = {
        'transaction': len(feature_cols),
        'user': 2,
        'merchant': 2
    }
    
    hidden_channels = cfg['model'].get('hidden_channels', 256)
    num_layers = cfg['model'].get('num_layers', 2)
    aggr = cfg['model'].get('aggr', 'mean')
    dropout = float(cfg['model'].get('dropout', 0.2))
    
    model = HeteroGraphSAGE(
        hidden_channels=hidden_channels,
        out_channels=1,
        num_layers=num_layers,
        data_metadata=train_data.metadata(),
        feature_dims=feature_dims,
        aggr=aggr,
        dropout=dropout
    ).to(device)
    
    # Lazy init parameter pass
    print("Inisialisasi lazy weights HeteroGraphSAGE...")
    with torch.no_grad():
        for sample_batch in train_loader:
            sample_batch = sample_batch.to(device)
            sample_num_nodes_dict = {ntype: sample_batch[ntype].num_nodes for ntype in sample_batch.node_types}
            _ = model(sample_batch.x_dict, sample_batch.edge_index_dict, num_nodes_dict=sample_num_nodes_dict)
            break
            
    num_pos = int(train_df_sampled['Is Fraud?'].sum())
    num_neg = int(len(train_df_sampled) - num_pos)
    raw_pos_weight = float(num_neg / (num_pos + 1e-7))
    
    # Strategi peredaman bobot kelas (v5 Golden Balance)
    loss_weight_strategy = cfg.get('training', {}).get('loss_weight_strategy', 'sqrt')
    if loss_weight_strategy == 'sqrt':
        eff_pos_weight = float(np.sqrt(raw_pos_weight))
    elif loss_weight_strategy == 'clipped':
        max_pw = float(cfg.get('training', {}).get('max_pos_weight', 50.0))
        eff_pos_weight = float(min(raw_pos_weight, max_pw))
    else:  # 'full'
        eff_pos_weight = float(raw_pos_weight)
        
    pos_weight = torch.tensor([eff_pos_weight]).to(device)
    print(f"GNN Class-Weight pos_weight ({loss_weight_strategy}): {pos_weight.item():.2f} (raw ratio: {raw_pos_weight:.2f})")
    print(f"GNN Architecture Regularization: Dropout = {dropout:.2f}")
    
    lr = float(cfg['training'].get('learning_rate', 0.001))
    weight_decay = float(cfg['training'].get('weight_decay', 1e-4))
    label_smoothing = float(cfg.get('training', {}).get('label_smoothing', 0.0))
    grad_clip_norm = float(cfg.get('training', {}).get('grad_clip_norm', 5.0))
    warmup_epochs = int(cfg.get('training', {}).get('warmup_epochs', 0))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    print(f"Optimizer: Adam (lr={lr}, weight_decay={weight_decay}) | Label Smoothing: {label_smoothing} | Grad Clip: {grad_clip_norm} | Warmup Epochs: {warmup_epochs}")

    # Siapkan Validation Data & Loader (dibangun sekali untuk evaluasi per-epoch & kalibrasi threshold)
    print("Membangun Graf Validation untuk evaluasi per-epoch...")
    val_graph_df = pd.concat([train_df, val_df], ignore_index=True)
    val_data = build_hetero_graph(val_graph_df, feature_cols)
    val_node_indices = torch.arange(len(train_df), len(val_graph_df))
    val_loader = get_neighbor_loader(
        val_data,
        batch_size=4096,
        num_neighbors=num_neighbors,
        method=sampling_method,
        shuffle=False,
        input_nodes=('transaction', val_node_indices)
    )

    # Learning Rate Scheduler (ReduceLROnPlateau berbasis Val AUPRC)
    use_scheduler = cfg.get('training', {}).get('use_lr_scheduler', True)
    if use_scheduler:
        lr_factor = float(cfg.get('training', {}).get('lr_scheduler_factor', 0.5))
        lr_patience = int(cfg.get('training', {}).get('lr_scheduler_patience', 5))
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='max', 
            factor=lr_factor, 
            patience=lr_patience,
            min_lr=1e-5
        )
        print(f"LR Scheduler: ReduceLROnPlateau (factor={lr_factor}, patience={lr_patience}, monitor='val_auprc')")
    else:
        scheduler = None
        
    early_stopping_patience = int(cfg.get('training', {}).get('early_stopping_patience', 15))

    # 7. Phase 1: Pelatihan GraphSAGE Encoder & Classifier Head
    epochs = cfg['training'].get('epochs', 100)
    print(f"\n--- [Phase 1: Training HeteroGraphSAGE ({epochs} Epochs dengan Best Checkpointing)] ---")
    gnn_history = {
        'epochs': [],
        'learning_rates': [],
        'train_loss': [], 'train_loss_unweighted': [], 'train_accuracy': [], 'train_f1': [], 'train_precision': [], 'train_recall': [], 'train_auprc': [], 'train_roc_auc': [],
        'val_loss': [], 'val_loss_unweighted': [], 'val_accuracy': [], 'val_f1': [], 'val_precision': [], 'val_recall': [], 'val_auprc': [], 'val_roc_auc': [],
        'val_optimal_tau': [], 'val_calibrated_accuracy': [], 'val_calibrated_f1': [], 'val_calibrated_precision': [], 'val_calibrated_recall': []
    }
    
    gnn_step_history = {
        'steps': [],
        'epochs': [],
        'train_loss': [],
        'train_loss_unweighted': [],
        'train_accuracy': []
    }
    global_step = 0
    log_step_interval = int(cfg.get('training', {}).get('log_step_interval', 1))
    
    best_val_auprc = -1.0
    best_epoch = 1
    best_model_state = None
    patience_counter = 0
    best_model_path = model_dir / f"exp23_{sampling_method}_best_gnn.pt"

    gnn_train_start = time.time()
    num_batches = len(train_loader)
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_loss_unw = 0.0
        total_samples = 0
        epoch_probs = []
        epoch_targets = []
        for batch_idx, batch in enumerate(train_loader):
            # Linear LR Warmup jika diaktifkan
            if epoch < warmup_epochs and num_batches > 0:
                warmup_factor = 0.1 + 0.9 * ((batch_idx + 1) / num_batches)
                for pg in optimizer.param_groups:
                    pg['lr'] = lr * warmup_factor

            batch = batch.to(device)
            optimizer.zero_grad()
            
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            
            loss = manual_weighted_bce(probs, targets, pos_weight, label_smoothing=label_smoothing)
            loss_unw = manual_unweighted_bce(probs, targets)
            loss.backward()
            
            # Gradient clipping untuk mencegah ledakan sampling variance dari super-hubs
            if grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
                
            optimizer.step()
            
            global_step += 1
            
            # Logging per-step mini-batch
            if global_step % log_step_interval == 0:
                with torch.no_grad():
                    step_preds = (probs >= 0.5).long()
                    step_acc = float((step_preds == targets.long()).float().mean().item())
                gnn_step_history['steps'].append(global_step)
                gnn_step_history['epochs'].append(epoch + 1)
                gnn_step_history['train_loss'].append(float(loss.item()))
                gnn_step_history['train_loss_unweighted'].append(float(loss_unw.item()))
                gnn_step_history['train_accuracy'].append(step_acc)
            
            total_loss += loss.item() * batch_size_current
            total_loss_unw += loss_unw.item() * batch_size_current
            total_samples += batch_size_current
            epoch_probs.append(probs.detach().cpu())
            epoch_targets.append(targets.detach().cpu())

        train_loss = total_loss / max(1, total_samples)
        train_loss_unw = total_loss_unw / max(1, total_samples)
        
        train_probs_cat = torch.cat(epoch_probs).numpy().ravel()
        train_targets_cat = torch.cat(epoch_targets).numpy().ravel().astype(np.int64)
        train_preds = (train_probs_cat >= 0.5).astype(int)
        
        train_acc = float(accuracy_score(train_targets_cat, train_preds))
        train_f1 = float(f1_score(train_targets_cat, train_preds, zero_division=0))
        train_prec = float(precision_score(train_targets_cat, train_preds, zero_division=0))
        train_rec = float(recall_score(train_targets_cat, train_preds, zero_division=0))
        train_auprc = float(average_precision_score(train_targets_cat, train_probs_cat)) if len(np.unique(train_targets_cat)) > 1 else 0.0
        train_roc_auc = float(roc_auc_score(train_targets_cat, train_probs_cat)) if len(np.unique(train_targets_cat)) > 1 else 0.0
        
        # Reset learning rate setelah warmup epoch
        if epoch < warmup_epochs:
            for pg in optimizer.param_groups:
                pg['lr'] = lr

        # Evaluasi validation set di akhir epoch
        val_metrics = evaluate_gnn(model, val_loader, device, pos_weight)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Checkpoint Best Model berbasis Validation AUCPR
        if val_metrics['auprc'] > best_val_auprc:
            best_val_auprc = val_metrics['auprc']
            best_epoch = epoch + 1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(model.state_dict(), best_model_path)
            patience_counter = 0
            improved_mark = "★ BEST"
        else:
            patience_counter += 1
            improved_mark = ""
            
        if scheduler is not None and epoch >= (warmup_epochs - 1):
            scheduler.step(val_metrics['auprc'])
        
        # Logging history
        gnn_history['epochs'].append(epoch + 1)
        gnn_history['learning_rates'].append(float(current_lr))
        gnn_history['train_loss'].append(train_loss)
        gnn_history['train_loss_unweighted'].append(train_loss_unw)
        gnn_history['train_accuracy'].append(train_acc)
        gnn_history['train_f1'].append(train_f1)
        gnn_history['train_precision'].append(train_prec)
        gnn_history['train_recall'].append(train_rec)
        gnn_history['train_auprc'].append(train_auprc)
        gnn_history['train_roc_auc'].append(train_roc_auc)
        
        gnn_history['val_loss'].append(val_metrics['loss'])
        gnn_history['val_loss_unweighted'].append(val_metrics['loss_unweighted'])
        gnn_history['val_accuracy'].append(val_metrics['accuracy'])
        gnn_history['val_f1'].append(val_metrics['f1_score'])
        gnn_history['val_precision'].append(val_metrics['precision'])
        gnn_history['val_recall'].append(val_metrics['recall'])
        gnn_history['val_auprc'].append(val_metrics['auprc'])
        gnn_history['val_roc_auc'].append(val_metrics['roc_auc'])
        gnn_history['val_optimal_tau'].append(float(val_metrics['optimal_tau']))
        gnn_history['val_calibrated_accuracy'].append(float(val_metrics['calibrated_accuracy']))
        gnn_history['val_calibrated_f1'].append(float(val_metrics['calibrated_f1']))
        gnn_history['val_calibrated_precision'].append(float(val_metrics['calibrated_precision']))
        gnn_history['val_calibrated_recall'].append(float(val_metrics['calibrated_recall']))
        
        print(f"Epoch {epoch+1:02d}/{epochs:02d} (lr: {current_lr:.6f}) {improved_mark}:")
        print(f"  • Train   | Loss(W): {train_loss:.4f} | Loss(U): {train_loss_unw:.4f} | Acc: {train_acc*100:.2f}% | F1: {train_f1:.4f} | Rec: {train_rec*100:.2f}% | Prec: {train_prec*100:.2f}% | AUCPR: {train_auprc:.4f} | ROC-AUC: {train_roc_auc:.4f}")
        print(f"  • Val@0.5 | Loss(W): {val_metrics['loss']:.4f} | Loss(U): {val_metrics['loss_unweighted']:.4f} | Acc: {val_metrics['accuracy']*100:.2f}% | F1: {val_metrics['f1_score']:.4f} | Rec: {val_metrics['recall']*100:.2f}% | Prec: {val_metrics['precision']*100:.2f}%")
        print(f"  • Val@tau*| tau*: {val_metrics['optimal_tau']:.3f} | Acc: {val_metrics['calibrated_accuracy']*100:.2f}% | F1: {val_metrics['calibrated_f1']:.4f} | Rec: {val_metrics['calibrated_recall']*100:.2f}% | Prec: {val_metrics['calibrated_precision']*100:.2f}% | AUCPR: {val_metrics['auprc']:.4f} | ROC-AUC: {val_metrics['roc_auc']:.4f}")
        
        # Early Stopping check
        if early_stopping_patience > 0 and patience_counter >= early_stopping_patience:
            print(f"\n[EARLY STOPPING] Pelatihan dihentikan lebih awal pada epoch {epoch+1}.")
            break

    gnn_train_time = time.time() - gnn_train_start
    print(f"Pelatihan GNN selesai dalam {gnn_train_time:.2f} detik.")
    
    # Simpan model GNN state dict terakhir
    gnn_model_path = model_dir / f"exp23_{sampling_method}_gnn.pt"
    torch.save(model.state_dict(), gnn_model_path)
    print(f"Model GNN (last epoch) tersimpan di: {gnn_model_path}")
    print(f"Model GNN (best checkpoint Epoch {best_epoch}, AUCPR={best_val_auprc:.4f}) tersimpan di: {best_model_path}")

    # Pulihkan (restore) bobot model GNN terbaik untuk inferensi evaluasi final
    if best_model_state is not None:
        print(f"\n[RESTORE CHECKPOINT] Memulihkan bobot terbaik GNN dari Epoch {best_epoch} (Val AUCPR: {best_val_auprc:.4f}) untuk evaluasi murni...")
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})

    # Bebaskan memori train
    del train_loader, train_data, optimizer
    if 'best_model_state' in locals():
        del best_model_state
    gc.collect()
    torch.cuda.empty_cache()

    # 8. Phase 2: Direct Inductive Evaluation & Dynamic Threshold Calibration (Pure GNN)
    print("\n--- [Phase 2: Kalibrasi Threshold Dinamis & Evaluasi Test Induktif Pure GNN] ---")
    
    # 8a. Prediksi langsung pada Validation set menggunakan GNN terbaik
    print("Menghitung probabilitas prediksi GNN pada Validation set...")
    val_probs, val_labels, inf_time_val_gnn = predict_gnn(model, val_loader, device)
    best_tau, val_tuning_stats = find_optimal_threshold(val_probs, val_labels)
    print(f"Optimal Threshold (tau*) dari Validation: {best_tau:.3f} (Val F1: {val_tuning_stats['val_f1']:.4f})")
    
    del val_loader, val_data, val_graph_df
    gc.collect()
    torch.cuda.empty_cache()

    # 8b. Membangun graf Test induktif (Train + Val + Test) & Inferensi Test
    print(f"\nMembangun Graf Test Induktif ({len(test_df)} transaksi uji)...")
    test_graph_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    test_data = build_hetero_graph(test_graph_df, feature_cols)
    test_node_indices = torch.arange(len(train_df) + len(val_df), len(test_graph_df))
    
    test_loader = get_neighbor_loader(
        test_data,
        batch_size=4096,
        num_neighbors=num_neighbors,
        method=sampling_method,
        shuffle=False,
        input_nodes=('transaction', test_node_indices)
    )
    
    print("Menghitung probabilitas prediksi GNN murni pada Test set...")
    test_probs, test_labels, inf_time_test_gnn = predict_gnn(model, test_loader, device)
    
    del test_data, test_loader, test_graph_df, model
    gc.collect()
    torch.cuda.empty_cache()

    # 8c. Evaluasi Metrik pada Test Set
    # Metrik @ Default Threshold 0.50
    preds_05 = (test_probs >= 0.5).astype(int)
    f1_05 = f1_score(test_labels, preds_05, zero_division=0)
    recall_05 = recall_score(test_labels, preds_05, zero_division=0)
    precision_05 = precision_score(test_labels, preds_05, zero_division=0)
    accuracy_05 = accuracy_score(test_labels, preds_05)
    
    # Metrik @ Optimal Threshold tau*
    preds_tau = (test_probs >= best_tau).astype(int)
    f1_tau = f1_score(test_labels, preds_tau, zero_division=0)
    f1_macro_tau = f1_score(test_labels, preds_tau, average='macro', zero_division=0)
    recall_tau = recall_score(test_labels, preds_tau, zero_division=0)
    precision_tau = precision_score(test_labels, preds_tau, zero_division=0)
    accuracy_tau = accuracy_score(test_labels, preds_tau)
    
    cm = confusion_matrix(test_labels, preds_tau)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        gmean = float(np.sqrt(recall_tau * specificity))
    else:
        gmean = 0.0

    # Metrik Independen Threshold
    roc_auc = roc_auc_score(test_labels, test_probs) if len(np.unique(test_labels)) > 1 else 0.0
    auprc = average_precision_score(test_labels, test_probs) if len(np.unique(test_labels)) > 1 else 0.0

    # Metrik Operasional Perbankan: Lift@1% dan TP@500
    percentile = 0.01
    top_1pct_count = max(1, int(round(len(test_labels) * percentile)))
    top_1pct_idx = np.argsort(test_probs)[::-1][:top_1pct_count]
    precision_1pct = float(test_labels[top_1pct_idx].mean())
    base_fraud_rate = float(test_labels.mean())
    lift_1_percent = (precision_1pct / base_fraud_rate) if base_fraud_rate > 0 else 0.0

    top_500_count = min(500, len(test_labels))
    top_500_idx = np.argsort(test_probs)[::-1][:top_500_count]
    tp_at_500 = int(test_labels[top_500_idx].sum())
    precision_at_500 = float(tp_at_500 / top_500_count)

    total_pipeline_time = time.time() - total_pipeline_start

    # Tampilkan Ringkasan Hasil
    print("\n" + "="*75)
    print("                      HASIL EVALUASI EKSPERIMEN 23")
    print("             (PURE END-TO-END HETEROGRAPHSAGE CLASSIFIER)")
    print("="*75)
    print(f"Protokol Waktu      : SHINE FRAUD II (Train: {train_days}d, Val: {val_days}d, Gap: {gap_days}d, Test: {test_days}d)")
    print(f"Sampling Method     : {sampling_method.upper()}")
    print(f"Optimal Threshold   : tau* = {best_tau:.3f} (Validation F1 = {val_tuning_stats['val_f1']:.4f})")
    print("-"*75)
    print(f"Metrik @ Threshold 0.5 (Default Baseline):")
    print(f"  • F1-Score        : {f1_05:.4f}")
    print(f"  • Recall          : {recall_05:.4f} ({recall_05*100:.2f}%)")
    print(f"  • Precision       : {precision_05:.4f} ({precision_05*100:.2f}%)")
    print(f"  • Accuracy        : {accuracy_05:.4f}")
    print("-"*75)
    print(f"Metrik @ Optimal Threshold tau* = {best_tau:.3f} (Rekomendasi Utama Pure GNN):")
    print(f"  • F1-Score (Bin)  : {f1_tau:.4f}")
    print(f"  • F1-Score (Macro): {f1_macro_tau:.4f}")
    print(f"  • Recall          : {recall_tau:.4f} ({recall_tau*100:.2f}%)")
    print(f"  • Precision       : {precision_tau:.4f} ({precision_tau*100:.2f}%)")
    print(f"  • Accuracy        : {accuracy_tau:.4f}")
    print(f"  • GMean           : {gmean:.4f}")
    print("-"*75)
    print(f"Metrik Bebas Threshold & Metrik Operasional:")
    print(f"  • AUCPR (AUPRC)   : {auprc:.4f}")
    print(f"  • ROC-AUC         : {roc_auc:.4f}")
    print(f"  • Lift@1%         : {lift_1_percent:.4f}x")
    print(f"  • TP@500          : {tp_at_500}/{top_500_count} ({precision_at_500*100:.2f}%)")
    print("-"*75)
    print(f"Efisiensi Waktu Komputasi (Pure GNN Tanpa XGBoost):")
    print(f"  • GNN Training    : {gnn_train_time:.2f} detik")
    print(f"  • GNN Val Tuning  : {inf_time_val_gnn:.2f} detik")
    print(f"  • GNN Test Infer  : {inf_time_test_gnn:.2f} detik")
    print(f"  • Total Pipeline  : {total_pipeline_time:.2f} detik ({total_pipeline_time/60:.2f} menit)")
    print("="*75)

    # 9. Simpan Hasil ke Berkas JSON
    result_dict = {
        "experiment_id": "EXP23",
        "experiment_name": "Pure End-to-End HeteroGraphSAGE Classification (SHINE FRAUD II - XGBoost Ablation)",
        "sampling_method": sampling_method,
        "time_window": {
            "reference_date": reference_date,
            "train_days": train_days,
            "val_days": val_days,
            "gap_days": gap_days,
            "test_days": test_days
        },
        "threshold_calibration": {
            "optimal_tau": float(best_tau),
            "val_f1": float(val_tuning_stats['val_f1']),
            "val_precision": float(val_tuning_stats['val_precision']),
            "val_recall": float(val_tuning_stats['val_recall'])
        },
        "metrics_default_threshold_05": {
            "threshold": 0.5,
            "f1_score": float(f1_05),
            "recall": float(recall_05),
            "precision": float(precision_05),
            "accuracy": float(accuracy_05)
        },
        "metrics_optimal_threshold": {
            "threshold": float(best_tau),
            "f1_score": float(f1_tau),
            "f1_macro": float(f1_macro_tau),
            "recall": float(recall_tau),
            "precision": float(precision_tau),
            "accuracy": float(accuracy_tau),
            "gmean": float(gmean)
        },
        "metrics_threshold_independent": {
            "auprc": float(auprc),
            "roc_auc": float(roc_auc),
            "lift_1_percent": float(lift_1_percent),
            "tp_at_500": int(tp_at_500),
            "precision_at_500": float(precision_at_500)
        },
        "runtime_seconds": {
            "data_preprocessing": float(data_prep_time),
            "gnn_training": float(gnn_train_time),
            "gnn_val_tuning": float(inf_time_val_gnn),
            "gnn_test_inference": float(inf_time_test_gnn),
            "total_test_inference": float(inf_time_test_gnn),
            "total_pipeline": float(total_pipeline_time)
        },
        "hyperparameters": {
            "hidden_channels": hidden_channels,
            "num_layers": num_layers,
            "aggr": aggr,
            "dropout": dropout,
            "gnn_epochs": epochs,
            "gnn_batch_size": batch_size,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "label_smoothing": label_smoothing,
            "grad_clip_norm": grad_clip_norm,
            "warmup_epochs": warmup_epochs,
            "loss_weight_strategy": loss_weight_strategy,
            "effective_pos_weight": float(pos_weight.item()),
            "best_gnn_epoch": int(best_epoch),
            "best_gnn_val_auprc": float(best_val_auprc),
            "num_neighbors": num_neighbors,
            "undersampling_ratio": undersample_ratio,
            "classifier": "Pure Linear Head + Sigmoid (No XGBoost)"
        }
    }
    
    result_path = result_dir / f"exp23_{sampling_method}_results.json"
    with open(result_path, 'w') as f:
        json.dump(result_dict, f, indent=4)
    print(f"\nHasil evaluasi lengkap tersimpan ke: {result_path}")

    # Simpan array prediksi mentah Test set untuk visualisasi kurva kontinu (PR, ROC, & Sensitivity)
    pred_path = result_dir / f"exp23_{sampling_method}_predictions.npz"
    np.savez_compressed(
        pred_path,
        test_probs=test_probs.astype(np.float32),
        test_labels=test_labels.astype(np.int8)
    )
    print(f"Prediksi test set mentah tersimpan ke: {pred_path}")

    history_dict = {
        "experiment_id": "EXP23",
        "sampling_method": sampling_method,
        "gnn_epochs": epochs,
        "best_epoch": int(best_epoch),
        "best_val_auprc": float(best_val_auprc),
        "gnn_history": gnn_history,
        "gnn_step_history": gnn_step_history
    }
    history_path = result_dir / f"exp23_{sampling_method}_history.json"
    with open(history_path, 'w') as f:
        json.dump(history_dict, f, indent=4)
    print(f"Riwayat training per-epoch lengkap tersimpan ke: {history_path}")

if __name__ == '__main__':
    main()



In [ ]:
%%writefile run_all.py
import os
import subprocess
import sys
import yaml

def run_script(script_name, args=[]):
    print(f"\n{'='*70}")
    print(f"  >>> MENJALANKAN SCRIPT: {script_name} {' '.join(args)}")
    print(f"{'='*70}\n")
    
    command = [sys.executable, script_name] + args
    result = subprocess.run(command)
    
    if result.returncode != 0:
        print(f"\n[ERROR] Eksekusi {script_name} gagal dengan kode {result.returncode}!")
        sys.exit(result.returncode)
    else:
        print(f"\n[SUCCESS] {script_name} selesai dieksekusi.\n")

if __name__ == "__main__":
    print("="*70)
    print("  MEMULAI EKSPERIMEN 23 v2: PURE END-TO-END HETEROGRAPHSAGE (TANPA XGBOOST)")
    print("  (256-dim, [25, 10] Neighbors, Early Stopping 15 Epochs)")
    print("="*70)
    
    # 1. Install dependencies jika di environment Kaggle/Colab
    if os.path.exists("/kaggle/input") or os.path.exists("/content"):
        print("\n[INFO] Menginstall dependencies (torch_geometric & pyg-lib)...")
        try:
            import torch
            pt_version = torch.__version__.split('+')[0]
            cuda_version = torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu'
            if cuda_version != 'cpu':
                url = f"https://data.pyg.org/whl/torch-{pt_version}+cu{cuda_version}.html"
                os.system(f"pip install torch_geometric pyg_lib torch_scatter torch_sparse -f {url} > /dev/null 2>&1")
            else:
                os.system("pip install torch_geometric > /dev/null 2>&1")
        except Exception as e:
            print(f"[WARNING] Fallback install torch_geometric: {e}")
            os.system("pip install torch_geometric > /dev/null 2>&1")

    # 2. Deteksi path konfigurasi
    if os.path.exists("/kaggle/input"):
        config_path = "config.kaggle.yaml"
        # Deteksi otomatis CSV di Kaggle
        csv_path = None
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'credit_card_transactions-ibm_v2.csv' in files:
                csv_path = os.path.join(root, 'credit_card_transactions-ibm_v2.csv')
                break
        if csv_path:
            print(f"[INFO] Dataset ditemukan di: {csv_path}")
            with open(config_path, 'r') as f:
                cfg = yaml.safe_load(f)
            cfg['data']['transactions'] = csv_path
            with open(config_path, 'w') as f:
                yaml.dump(cfg, f)
    elif os.path.exists("/content"):
        config_path = "config.colab.yaml"
    else:
        config_path = "config.reference.yaml"

    # 3. Jalankan eksperimen untuk ketiga metode sampling
    sampling_methods = ["uniform", "topology", "importance"]
    for method in sampling_methods:
        print(f"\n{'*'*70}")
        print(f"  >>> MULAI EKSPERIMEN DENGAN SAMPLING: {method.upper()}")
        print(f"{'*'*70}\n")
        run_script("main.py", ["--config", config_path, "--sampling", method])
        
    # 4. Generate Visualisasi Hasil (16 Chart Ilmiah)
    if os.path.exists("visualize_results.py"):
        print("\n" + "="*70)
        print("  >>> MENJALANKAN GENERASI VISUALISASI HASIL PURE GNN (CHART 1-17)")
        print("="*70 + "\n")
        try:
            with open(config_path, 'r') as f:
                cfg_loaded = yaml.safe_load(f)
            res_dir = cfg_loaded.get('output', {}).get('result_dir', '../../result/exp23/v2')
            plots_dir = os.path.join(res_dir, 'plots')
            run_script("visualize_results.py", ["--result-dir", res_dir, "--output-dir", plots_dir])
        except Exception as e:
            print(f"[WARNING] Gagal menjalankan visualisasi otomatis: {e}")

    print("\n" + "="*70)
    print("  🎉 SELURUH PROSES PELATIHAN, EVALUASI, DAN VISUALISASI EXP23 SELESAI! 🎉")
    print("="*70)



In [ ]:
!python run_all.py